Code for generating simulated data for the Bronze layer tables. These are slowly changing datasets:
- Customers
- Addresses
- Products

Initialization steps  

In [0]:
%pip install faker

In [0]:
%pip install faker faker-commerce

In [0]:
volume_path ="/Volumes/workspace/default/amazon_fullfilment/source/"

In [0]:
catalog_name = "workspace"
database_name = "amazon_fullfilment_bronze"

In [0]:
bronze_volume_path ="/Volumes/workspace/amazon_fullfilment_bronze/landing_zone/"
source_volume_path = "/Volumes/workspace/default/amazon_fullfilment/source/"

Generating source data

In [0]:
#Generate Customers and addresses data

from pyspark.sql import Row
from faker import Faker
import random

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType

# Initialize Faker with Canadian English locale
fake = Faker('en_CA')

def generate_customer_bundle(num_customers=100):
    customers = []
    addresses = []
    
    for _ in range(num_customers):
        # 1. Create unique Customer
        cust_id = str(fake.uuid4())
        customers.append(Row(
            customer_id=cust_id,
            first_name=fake.first_name(),
            last_name=fake.last_name(),
            gender=random.choice(["Male", "Female"]),
            email=fake.email(),
            phone=fake.phone_number()
        ))
        
        # 2. Create 1 to 3 addresses per Customer
        num_addr = random.randint(1, 3)
        for _ in range(num_addr):
            addresses.append(Row(
                address_id=str(fake.uuid4()),
                customer_id=cust_id, # Linking key
                street=fake.street_address(),
                city=fake.city(),
                province=fake.province_abbr(),
                postal_code=fake.postalcode()
            ))
            
    return spark.createDataFrame(customers), spark.createDataFrame(addresses)

# Generate the dataframes
customers_df, addresses_df = generate_customer_bundle(500)



# Save to source folder
(customers_df.write
 .format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/customers"))
(addresses_df
 .write.format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/addresses"))





In [0]:
#Generate products data

from pyspark.sql import Row
from faker import Faker
import faker_commerce
import random

fake = Faker('en_CA')
fake.add_provider(faker_commerce.Provider)

# 1. Define the Semantic Map
# This ensures that 'Electronics' only gets electronic-sounding names
PRODUCT_MAP = {
    "Electronics": {
        "brands": ["TechFlow", "ElectroNova", "VoltPulse", "AmazonBasics"],
        "items": ["Wireless Mouse", "Noise-Cancelling Headphones", "4K Monitor", "USB-C Hub", "Smart Watch"]
    },
    "Home & Kitchen": {
        "brands": ["PureComfort", "ChefMaster", "HearthStone", "AmazonBasics"],
        "items": ["Air Fryer", "Memory Foam Pillow", "Cast Iron Skillet", "Electric Kettle", "Microfiber Towel"]
    },
    "Groceries": {
        "brands": ["FreshPick", "Nature's Own", "OrganicHarvest", "WholeFoods"],
        "items": ["Artisanal Bacon", "Organic Honey", "Maple Syrup", "Cold Brew Coffee", "Aged Cheddar"]
    },
    "Music": {
        "brands": ["AudioWave", "EchoBeats", "MelodyMaker"],
        "items": ["Vinyl Record Player", "Acoustic Guitar Strings", "Studio Microphone", "Drum Sticks"]
    }
}

def generate_logical_products(num_products=500):
    products = []
    categories = list(PRODUCT_MAP.keys())
    
    for _ in range(num_products):
        # 2. Pick the Category first
        category = random.choice(categories)
        
        # 3. Use the Category to pick matching brands and items
        brand = random.choice(PRODUCT_MAP[category]["brands"])
        item = random.choice(PRODUCT_MAP[category]["items"])
        
        product_name = f"{brand} {item}"
        
        if random.random() < 0.7:
                 price = round(random.uniform(5.00, 50.00), 2)
        else:
                 price = round(random.uniform(50.01, 800.00), 2)
                 
        products.append(Row(
            product_id=str(fake.uuid4()),
            sku=f"{category[:3].upper()}-{random.randint(100, 999)}",
            product_name=product_name,
            category=category,
            #price=float(fake.ecommerce_price()),
            weight_kg=round(random.uniform(0.1, 10.0), 2)
        ))
    
    return spark.createDataFrame(products)

# Test it
products_df = generate_logical_products()
(products_df.write
 .format("csv")
 .option("header",True)
 .option("delimiter",",")
 .mode("append")
 .save(f"{volume_path}/products"))



In [0]:
# Insert into Bronze layer customer and address tables
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType


# Define exactly what the data should look like
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True)
])

address_schema = StructType([
    StructField("address_id", StringType(),True),
    StructField("customer_id", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("province", StringType(), True),
    StructField("postal_code", StringType(), True)
])


# 1. Define your paths

bronze_customer_path = f"{bronze_volume_path}customers"
bronze_address_path = f"{bronze_volume_path}addresses"

# 2. Set up the Auto Loader Stream for customers
bronze_customers_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true") 
  .option("cloudFiles.schemaLocation", bronze_customer_path) 
  .schema(customer_schema)
  .load(f"{source_volume_path}customers"))

# 3. Write it to a permanent Bronze Table for customers
(bronze_customers_stream.writeStream
  .option("checkpointLocation", bronze_customer_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.customers"))


# 4. Set up the Auto Loader Stream for addreses
bronze_addresses_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")  
  .option("cloudFiles.schemaLocation", bronze_address_path) 
  .schema(address_schema)
  .load(f"{source_volume_path}addresses"))

# 5. Write it to a permanent Bronze Table for addresses
(bronze_addresses_stream.writeStream
  .option("checkpointLocation", bronze_address_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.addresses"))

In [0]:
# Insert into Bronze layer Products table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType

# Define exactly what the data should look like
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("weight_kg", DoubleType(), True)
])


# 1. Define your paths
bronze_products_path = f"{bronze_volume_path}products"

# 2. Set up the Auto Loader Stream
bronze_products_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv") # Assuming you save mock data as Delta
  .option("header", "true") 
  .option("cloudFiles.schemaLocation", bronze_products_path) 
  .schema(products_schema)
  .load(f"{source_volume_path}products"))

# 3. Write it to a permanent Bronze Table
(bronze_products_stream.writeStream
  .option("checkpointLocation", bronze_products_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.products"))

# Processing into Silver layer

In [0]:
# Insert into Silver layer customers table - SCD1

from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable 

# 1. Read from the Bronze Streaming Table
bronze_customer_stream = spark.readStream \
    .table("workspace.amazon_fullfilment_bronze.customers")

def upsert_to_silver_customer(batch_df, batch_id):
    # 2. Filter for Quality: Ensure customer_id is NOT NULL
    # Records failing this will be skipped (dropped)
    clean_batch_df = batch_df.filter(
        col("customer_id").isNotNull()
    ).withColumn("Updated_Timestamp", current_timestamp())

    silver_table_customer = DeltaTable.forName(spark, "workspace.amazon_fullfilment_silver.customer_silver")
    # 3. Perform the SCD Type 1 Merge (Overwrite on ID match)
    (silver_table_customer.alias("target")
        .merge(
            clean_batch_df.alias("source"),
            "source.customer_id = target.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


# 4. Start the Stream
query = (bronze_customer_stream.writeStream
    .foreachBatch(upsert_to_silver_customer)
    .option("checkpointLocation", "/Volumes/workspace/amazon_fullfilment_silver/processing_zone/customer_silver")
    .trigger(availableNow=True)
    .start())

In [0]:
# Insert into Silver layer adresses table - SCD1

from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable 

# 1. Read from the Bronze Streaming Table
bronze_address_stream = spark.readStream \
    .table("workspace.amazon_fullfilment_bronze.addresses")

def upsert_to_silver(batch_df, batch_id):
    # 1. Clean and deduplicate the batch
    # SCD1 will fail if the same customer/address combo appears twice in one batch
    clean_batch_df = batch_df.filter(
        col("city").isNotNull() & col("province").isNotNull()
    ).dropDuplicates(["customer_ID", "street", "province", "postal_code"]) \
     .withColumn("Updated_Timestamp", current_timestamp())

    # 2. Create a temporary local view for the SQL engine
    # Using a unique name per batch is safer in the Free Tier
    view_name = f"batch_address_{batch_id}"
    clean_batch_df.createOrReplaceTempView(view_name)

    target_table = "workspace.amazon_fullfilment_silver.address_silver"

    # 3. Use Native Spark SQL Merge
    batch_df.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING {view_name} AS source
        ON source.customer_ID = target.customer_ID 
           AND source.street = target.street_address 
           AND source.province = target.province 
           AND source.postal_code = target.postal_code
        
        WHEN MATCHED THEN
          UPDATE SET target.Address_ID = source.address_id, 
                     target.city = source.city, 
                     target.updated_timestamp = source.updated_timestamp
          
        WHEN NOT MATCHED THEN
          INSERT (address_id, customer_id, street_address, city, province, postal_code, updated_timestamp) VALUES (source.address_id, source.customer_ID, source.street, source.city, source.province, source.postal_code, source.updated_timestamp)
    """)
    
    # Clean up the view to free memory
    batch_df.sparkSession.catalog.dropTempView(view_name)

    # 4. Start the Stream
query = (bronze_address_stream.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", "/Volumes/workspace/amazon_fullfilment_silver/processing_zone/address_silver")
    .trigger(availableNow=True)
    .start())

In [0]:
# Insert into Silver layer products_silver table - SCD2

from pyspark.sql.functions import col, current_timestamp, lit, expr
from delta.tables import DeltaTable 

def upsert_to_silver_products(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # 1. Capture the correct session
    batch_spark = batch_df.sparkSession

    # 2. Quality Filter
    clean_batch_df = batch_df.filter(
        col("product_id").isNotNull() & 
        col("product_name").isNotNull() & 
        col("category").isNotNull()
    )

    silver_table = DeltaTable.forName(batch_spark, "workspace.amazon_fullfilment_silver.products_silver")

    # 3. IDENTIFY UPDATES: Join source with target to find records that already exist
    # These records will need to have their old versions "expired"
    new_records_to_insert = clean_batch_df.alias("updates") \
        .join(silver_table.toDF().alias("target"), "product_id") \
        .where("target.is_current = true AND (updates.product_name <> target.product_name OR updates.category <> target.category)") \
        .select("updates.*")

    # 4. PREPARE THE STAGED DATA: 
    # We want to insert ALL new records, plus the new versions of updated records
    # We set 'merge_key' to None for records that are just updates to force an insert
    staged_updates = clean_batch_df.withColumn("merge_key", col("product_id")) \
        .unionByName(new_records_to_insert.withColumn("merge_key", lit(None)))

    # 5. THE SCD2 MERGE
    (silver_table.alias("target")
        .merge(
            staged_updates.alias("source"),
            "target.product_id = source.merge_key"
        )
        .whenMatchedUpdate(
            condition="target.is_current = true AND (source.product_name <> target.product_name OR source.category <> target.category)",
            set={
                "is_current": "false",
                "end_date": "current_timestamp()",
                "Updated_Timestamp": "current_timestamp()"
            }
        )
        .whenNotMatchedInsert(
            values={
                "product_id": "source.product_id",
                "sku": "source.sku",
                "product_name": "source.product_name",
                "category": "source.category",
                "weight_kg": "source.weight_kg",
                "is_current": "true",
                "start_date": "current_timestamp()",
                "end_date": "to_timestamp('9999-12-31 00:00:00')",
                "Updated_Timestamp": "current_timestamp()"
            }
        )
        .execute())

# 4. Start the Stream (Ensure you read from PRODUCTS bronze, not customers)
bronze_products_stream = spark.readStream \
    .table("workspace.amazon_fullfilment_bronze.products") # Corrected table name

query = (bronze_products_stream.writeStream
    .foreachBatch(upsert_to_silver_products)
    .option("checkpointLocation", "/Volumes/workspace/amazon_fullfilment_silver/processing_zone/products_silver")
    .trigger(availableNow=True)
    .start())

In [0]:
%sql
alter table workspace.amazon_fullfilment_silver.products_silver
add constraint products_silver_pk Primary Key (product_id);